# A/B Testing and Power Analysis In-Depth
This notebook demonstrates how to design and analyze A/B tests using Python.

# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q statsmodels
    print('Packages installed.')


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

## 1. Power Analysis
Before running a test, we determine the required sample size to detect a specific effect size with high probability (Power).

In [ ]:
# Parameters
baseline_conversion = 0.10  # 10%
minimum_detectable_effect = 0.02 # Detect 12%
alpha = 0.05 # Significance level
power = 0.80 # Power (1 - Beta)

effect_size = sms.proportion_effectsize(baseline_conversion, baseline_conversion + minimum_detectable_effect)

sample_size = sms.NormalIndPower().solve_power(
    effect_size=effect_size,
    power=power,
    alpha=alpha,
    ratio=1
)

print(f"Required sample size per group: {int(np.ceil(sample_size))}")

## 2. Generate Synthetic Data
Simulating the experiment results.

In [ ]:
n = int(np.ceil(sample_size))
np.random.seed(42)

# Control Group (10% conversion)
control = np.random.binomial(n=1, p=baseline_conversion, size=n)

# Treatment Group (12.5% conversion - slightly better than target to ensure detection)
treatment = np.random.binomial(n=1, p=0.125, size=n)

df = pd.DataFrame({
    'group': ['control'] * n + ['treatment'] * n,
    'converted': np.concatenate([control, treatment])
})

print(df.groupby('group')['converted'].mean())

## 3. Analyze Results
We use a Z-test for proportions.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

control_conv = df[df['group'] == 'control']['converted']
treat_conv = df[df['group'] == 'treatment']['converted']

count = np.array([treat_conv.sum(), control_conv.sum()])
nobs = np.array([len(treat_conv), len(control_conv)])

z_stat, p_val = proportions_ztest(count, nobs, alternative='two-sided')

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_val:.4f}")

if p_val < 0.05:
    print("Result is Statistically Significant! Reject Null Hypothesis.")
else:
    print("Result is NOT Significant. Fail to reject Null Hypothesis.")